In [1]:
%pip install pypdf
%pip install sentence-transformers
%pip install qdrant-client
%pip install langchain langchain-experimental langchain-community

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
# ============================================================
# PDF RAG using Semantic Chunking + Sentence Transformers + Qdrant
# ============================================================

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
)
from langchain_experimental.text_splitter import SemanticChunker

# ============================================================
# Configuration
# ============================================================

PDF_PATH = "/home/linux00/Downloads/p/Resumes/GVyshnavi_.pdf"

COLLECTION_NAME = "semantic_rag"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

TOP_K = 3


# ============================================================
# Embedding Wrapper for SemanticChunker
# ============================================================

class SentenceTransformerEmbeddings:

    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()


# ============================================================
# PDF Reader
# ============================================================

def extract_text_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    content = ""

    for page in reader.pages:

        text = page.extract_text()

        if text:
            content += text + "\n"

    return content


# ============================================================
# Semantic Chunking
# ============================================================

def semantic_chunk_text(text, embedding_wrapper):

    splitter = SemanticChunker(
        embeddings=embedding_wrapper,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90,
    )

    documents = splitter.create_documents([text])

    chunks = [doc.page_content for doc in documents]

    return chunks


# ============================================================
# Embedding Model
# ============================================================

def load_embedding_model():

    return SentenceTransformer(EMBEDDING_MODEL)


# ============================================================
# Create Collection
# ============================================================

def create_collection(client, vector_size):

    try:
        client.delete_collection(COLLECTION_NAME)
        print(f"Deleted existing collection: {COLLECTION_NAME}")

    except Exception:
        pass

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE,
        ),
    )

    print(f"Created collection: {COLLECTION_NAME}")


# ============================================================
# Store Chunks
# ============================================================

def store_chunks(client, chunks, embeddings):

    points = []

    for idx, (chunk, vector) in enumerate(zip(chunks, embeddings)):

        points.append(
            PointStruct(
                id=idx,
                vector=vector.tolist(),
                payload={
                    "text": chunk
                },
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
    )

    print(f"Inserted {len(points)} chunks into Qdrant")


# ============================================================
# Search
# ============================================================

def search(client, model, query, top_k=3):

    query_vector = model.encode(query)

    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector.tolist(),
        limit=top_k,
    )

    return results


# ============================================================
# Main
# ============================================================

def main():

    print("\nReading PDF...")

    content = extract_text_from_pdf(PDF_PATH)

    print(f"Total Characters : {len(content)}")

    print("\nLoading embedding model...")

    model = load_embedding_model()

    embedding_wrapper = SentenceTransformerEmbeddings(model)

    print("\nSemantic Chunking...")

    chunks = semantic_chunk_text(
        content,
        embedding_wrapper,
    )

    print(f"\nTotal Semantic Chunks : {len(chunks)}")

    print("\nGenerated Chunks\n")

    for i, chunk in enumerate(chunks, start=1):

        print("=" * 80)
        print(f"Chunk {i}")
        print(chunk)
        print("=" * 80)

    print("\nGenerating embeddings...")

    embeddings = model.encode(
        chunks,
        show_progress_bar=True,
    )

    print("Embeddings generated")

    print("\nConnecting to Qdrant...")

    client = QdrantClient(
        host=QDRANT_HOST,
        port=QDRANT_PORT,
    )

    create_collection(
        client,
        vector_size=len(embeddings[0]),
    )

    store_chunks(
        client,
        chunks,
        embeddings,
    )

    print("\nRAG System Ready")

    query = "how many years of experience does candidate have"
    results = search(
        client,
        model,
        query,
        top_k=TOP_K,
    )

    print("\nTop Results")

    for rank, hit in enumerate(results, start=1):

        print("\n" + "=" * 80)

        print(f"Rank  : {rank}")
        print(f"Score : {hit.score:.4f}")

        print("\nRetrieved Chunk:\n")

        print(hit.payload)

        print("=" * 80)


if __name__ == "__main__":
    main()


Reading PDF...
Total Characters : 3591

Loading embedding model...

Semantic Chunking...

Total Semantic Chunks : 4

Generated Chunks

Chunk 1
Gudla Vyshnavi Email: gvyshnavi19@gmail.com
LinkedIn: www.linkedin.com/in/vyshnavig Mobile: +91-9381454255
Github: github.com/VyshnaviRG
Career Summary
• A dedicated Software Developer with experience in optimizing data processing and designing applications. Eager to contribute
to team success by leveraging Java, Spring frameworks, and database knowledge. Demonstrated ability to improve system
performance and streamline processes.
Chunk 2
Looking forward to applying and expanding skills in a collaborative and
growth-oriented environment. Experience
• AdMAVIN Chennai
AdMAVIN — Software Developer (Full-time) Sep 2022 - Feb 2023 & Sep 2023 - PRESENT
◦ Optimized mobility data processing by implementing multithreading and algorithmic solutions, resulting in a 25%
performance improvement. ◦ Engineered a navigation service utilizing geospatial data. ◦

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated

Connecting to Qdrant...
Deleted existing collection: semantic_rag
Created collection: semantic_rag
Inserted 4 chunks into Qdrant

RAG System Ready

Top Results

Rank  : 1
Score : 0.1314

Retrieved Chunk:

{'text': 'Looking forward to applying and expanding skills in a collaborative and\ngrowth-oriented environment. Experience\n• AdMAVIN Chennai\nAdMAVIN — Software Developer (Full-time) Sep 2022 - Feb 2023 & Sep 2023 - PRESENT\n◦ Optimized mobility data processing by implementing multithreading and algorithmic solutions, resulting in a 25%\nperformance improvement. ◦ Engineered a navigation service utilizing geospatial data. ◦ Developed and implemented algorithms for computing Out-of-Home (OOH) industry metrics. ◦ Constructed reusable Java libraries to enhance code maintainability and streamline development processes. ◦ Technologies: Java 17, JWT, Spring MVC, Microservices, REST, MongoDB, JSP, MySQL, Multithreading, OSM\n• AdMAVIN Chennai\nIEAC — Software Developer